In [7]:
!pip install aif360

In [3]:
!pip install ucimlrepo

In [10]:
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
from ucimlrepo import fetch_ucirepo

# Adult (protected = sex, privileged = Male, favorable = income >50K)
df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS,
                            "uciml/adult-census-income", "adult.csv")
df = df.replace("?", np.nan).dropna().sample(3000, random_state=42)
y_adult = (df["income"] == ">50K").astype(int)
X_adult = df.drop(columns=["income", "education", "native.country", "sex"])
X_adult = pd.get_dummies(X_adult, dtype=float)
X_adult["sex"] = (df["sex"] == "Male").astype(int)

# COMPAS (protected = race, privileged = Caucasian, favorable = no recidivism)
url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
df = pd.read_csv(url)
df = df[df.days_b_screening_arrest.between(-30, 30) & (df.is_recid != -1)
        & (df.c_charge_degree != "O") & (df.score_text != "N/A")]
df = df[df.race.isin(["Caucasian", "African-American"])]
X_compas = df[["age", "priors_count", "juv_fel_count", "juv_misd_count", "juv_other_count"]].copy()
X_compas["charge_felony"] = (df.c_charge_degree == "F").astype(int)
X_compas["sex_male"] = (df.sex == "Male").astype(int)
X_compas["race"] = (df.race == "Caucasian").astype(int)
y_compas = 1 - df["two_year_recid"]

# German (protected = age, privileged = 25 or older, favorable = good credit)
g = fetch_ucirepo(id=144)
Xg = g.data.features.copy()
y_german = (g.data.targets.iloc[:, 0] == 1).astype(int)
X_german = pd.get_dummies(Xg, dtype=float)
X_german["age_priv"] = (Xg["Attribute13"] >= 25).astype(int)

datasets = {
    "Adult":  (X_adult,  y_adult,  "sex"),
    "COMPAS": (X_compas, y_compas, "race"),
    "German": (X_german, y_german, "age_priv"),
}

/tmp/ipykernel_524/2554256426.py:8: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS,


Using Colab cache for faster access to the 'adult-census-income' dataset.


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing, LFR

def to_aif(X, y, prot):
    d = X.copy()
    d["label"] = np.asarray(y)
    return BinaryLabelDataset(df=d, label_names=["label"], protected_attribute_names=[prot])

def metrics(ds, prot):
    m = BinaryLabelDatasetMetric(ds, unprivileged_groups=[{prot: 0}], privileged_groups=[{prot: 1}])
    return abs(m.statistical_parity_difference()), m.disparate_impact(), m.consistency()[0]

def with_preds(ds, pred):
    out = ds.copy(deepcopy=True)
    out.labels = np.asarray(pred).reshape(-1, 1)
    return out

rows = []
for name, (X, y, prot) in datasets.items():
    X = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=X.columns, index=X.index)
    feats = [c for c in X.columns if c != prot]

    # before mitigation
    rows.append([name, "Original labels", *metrics(to_aif(X, y, prot), prot), np.nan])

    # stratified split
    strat = y.astype(str) + X[prot].astype(str)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=strat, random_state=42)
    train, test = to_aif(Xtr, ytr, prot), to_aif(Xte, yte, prot)

    # baseline, no mitigation
    pred = LogisticRegression(max_iter=1000).fit(Xtr[feats], ytr).predict(Xte[feats])
    rows.append([name, "LR baseline (test)", *metrics(with_preds(test, pred), prot),
                 accuracy_score(yte, pred)])

    # Reweighing on the training data, then the same classifier
    # this implementation is based upon the implementation on this site: https://www.geeksforgeeks.org/machine-learning/ensuring-fairness-in-machine-learning-algorithms/
    rw = Reweighing(unprivileged_groups=[{prot: 0}], privileged_groups=[{prot: 1}])
    weights = rw.fit_transform(train).instance_weights
    pred = LogisticRegression(max_iter=1000).fit(Xtr[feats], ytr, sample_weight=weights).predict(Xte[feats])
    rows.append([name, "Reweighing (test)", *metrics(with_preds(test, pred), prot),
                 accuracy_score(yte, pred)])

    # LFR from AIF360 (the algorithm from the paper), fitted on the training data
    for Az in [1, 5, 50]:
        lfr = LFR(unprivileged_groups=[{prot: 0}], privileged_groups=[{prot: 1}],
                  k=5, Ax=0.01, Ay=1.0, Az=Az, verbose=0)
        lfr.fit(train, maxiter=300, maxfun=300)
        pred = lfr.transform(test)
        rows.append([name, f"LFR Az={Az} (test)", *metrics(pred, prot),
                     accuracy_score(yte, pred.labels.ravel())])

print(pd.DataFrame(rows, columns=["Dataset", "Setting", "DP_Diff", "DIR", "Consistency", "Accuracy"])
      .round(3).to_string(index=False))

/usr/local/lib/python3.13/dist-packages/aif360/metrics/dataset_metric.py:82: RuntimeWarning: invalid value encountered in scalar divide
  return metric_fun(privileged=False) / metric_fun(privileged=True)
/usr/local/lib/python3.13/dist-packages/aif360/metrics/dataset_metric.py:82: RuntimeWarning: invalid value encountered in scalar divide
  return metric_fun(privileged=False) / metric_fun(privileged=True)
/usr/local/lib/python3.13/dist-packages/aif360/metrics/dataset_metric.py:82: RuntimeWarning: invalid value encountered in scalar divide
  return metric_fun(privileged=False) / metric_fun(privileged=True)
/usr/local/lib/python3.13/dist-packages/aif360/metrics/dataset_metric.py:82: RuntimeWarning: invalid value encountered in scalar divide
  return metric_fun(privileged=False) / metric_fun(privileged=True)


Dataset            Setting  DP_Diff   DIR  Consistency  Accuracy
  Adult    Original labels    0.190 0.384        0.814       NaN
  Adult LR baseline (test)    0.212 0.184        0.939     0.836
  Adult  Reweighing (test)    0.109 0.484        0.937     0.828
  Adult    LFR Az=1 (test)    0.000   NaN        1.000     0.753
  Adult    LFR Az=5 (test)    0.000   NaN        1.000     0.753
  Adult   LFR Az=50 (test)    0.000   NaN        1.000     0.753
 COMPAS    Original labels    0.132 0.783        0.658       NaN
 COMPAS LR baseline (test)    0.227 0.680        0.965     0.668
 COMPAS  Reweighing (test)    0.218 0.697        0.965     0.672
 COMPAS    LFR Az=1 (test)    0.002 0.998        0.999     0.530
 COMPAS    LFR Az=5 (test)    0.001 0.999        0.999     0.528
 COMPAS   LFR Az=50 (test)    0.000   NaN        1.000     0.471
 German    Original labels    0.129 0.821        0.742       NaN
 German LR baseline (test)    0.213 0.727        0.821     0.737
 German  Reweighing (test